# Pokémon Type Classifier
Multi-label classifier using official artwork from `pokemon_data.csv`.

## 1. Imports & Setup

In [14]:
import os
import csv
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
import requests
from io import BytesIO

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

Using device: cuda


## 2. Pokémon Types

In [15]:
POKEMON_TYPES = [
    "normal", "fire", "water", "grass", "electric", "ice",
    "fighting", "poison", "ground", "flying", "psychic",
    "bug", "rock", "ghost", "dark", "dragon", "steel", "fairy"
]

## 3. Dataset Class

In [16]:
class PokemonDataset(Dataset):
    def __init__(self, csv_file, transform=None, img_dir='pokemon_images'):
        self.rows = list(csv.DictReader(open(csv_file)))
        self.transform = transform
        self.img_dir = img_dir
        os.makedirs(img_dir, exist_ok=True)

    def load_sprite(self, url, local_name):
        if not url:
            return Image.new('RGB', (224, 224), (0, 0, 0))

        filepath = os.path.join(self.img_dir, local_name)
        if not os.path.exists(filepath):
            try:
                img_data = requests.get(url).content
                with open(filepath, 'wb') as f:
                    f.write(img_data)
            except:
                return Image.new('RGB', (224, 224), (0, 0, 0))

        img = Image.open(filepath)
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return img

    def encode_types(self, type_string):
        labels = type_string.split(", ")
        return torch.tensor([1 if t in labels else 0 for t in POKEMON_TYPES], dtype=torch.float32)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        url = row['official_artwork']
        image = self.load_sprite(url, f"{row['id']}.png")
        if self.transform:
            image = self.transform(image)
        return image, self.encode_types(row['types'])

## 4. Data Transforms & Dataset Split

In [17]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = PokemonDataset('pokemon_data.csv', transform)

# Split train/test
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")

Train size: 820, Test size: 205


## 5. Model Definition

In [18]:
def build_model():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(512, len(POKEMON_TYPES))
    return model

model = build_model().to(DEVICE)

## 6. Training Loop

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
EPOCHS = 100

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    for imgs, labels in train_loader:
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {running_loss:.4f}")

torch.save(model.state_dict(), 'pokemon_model.pt')
print('Model saved as pokemon_model.pt')

Epoch 1/5 - Loss: 23.6366
Epoch 2/5 - Loss: 10.9803
Epoch 3/5 - Loss: 7.5856
Epoch 4/5 - Loss: 4.8684
Epoch 5/5 - Loss: 3.2050
Model saved as pokemon_model.pt


## 7. Evaluation on Test Set

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)
        logits = model(imgs)
        probs = torch.sigmoid(logits)
        all_preds.append(probs)
        all_labels.append(labels)

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

# display results on test set per type
pred_labels = (all_preds > 0.5).float()
accuracy_per_type = (pred_labels == all_labels).float().mean(dim=0)

for t, acc in zip(POKEMON_TYPES, accuracy_per_type):
    print(f"{t:<10}: {acc:.3f}")

normal    : 0.898
fire      : 0.917
water     : 0.863
grass     : 0.878
electric  : 0.937
ice       : 0.946
fighting  : 0.912
poison    : 0.912
ground    : 0.917
flying    : 0.893
psychic   : 0.888
bug       : 0.898
rock      : 0.941
ghost     : 0.937
dark      : 0.927
dragon    : 0.922
steel     : 0.946
fairy     : 0.937


## 8. Prediction / Inference

In [21]:
def predict(model, image_path, threshold=0.3):
    from torchvision import transforms
    from PIL import Image
    import torch

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    img = Image.open(image_path)
    if img.mode != 'RGB':
        img = img.convert('RGB')

    img = transform(img).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(img)
        probs = torch.sigmoid(logits)[0]

    results = [(t, float(p)) for t, p in zip(POKEMON_TYPES, probs) if p > threshold]
    if not results:
        max_idx = torch.argmax(probs).item()
        results = [(POKEMON_TYPES[max_idx], float(probs[max_idx]))]

    return results

# Example usage
model.load_state_dict(torch.load('pokemon_model.pt', map_location=DEVICE))
image_path = 'test_images/fake.png'
predicted = predict(model, image_path)
print('Predicted types:', predicted)

Predicted types: [('grass', 0.5059615969657898)]
